In [1]:
import os
import pandas as pd
from openai import AsyncOpenAI
import json
import asyncio
from tqdm import tqdm
import time
import nest_asyncio
import psutil
import gc
from datetime import datetime
import logging
from typing import Set, Tuple

# 配置日志记录
logging.basicConfig(
    filename='news_analysis.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 允许嵌套事件循环
nest_asyncio.apply()

# 配置参数
DEEPSEEK_API_KEY = "sk-8e2d6d0d2972416d9e22c56925c71433"
OUTPUT_DIR = "analysis_results"
NEWS_DATA_DIR = "news_data"
MAX_CONCURRENT_REQUESTS = 15
REQUEST_TIMEOUT = 15
MEMORY_THRESHOLD = 90
BATCH_SIZE = 30
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 3, 1)

# 初始化异步客户端
aclient = AsyncOpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

def ensure_output_dir() -> None:
    """确保输出目录存在"""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_memory_usage() -> float:
    """获取当前内存使用百分比"""
    return psutil.virtual_memory().percent

def adjust_parameters(memory_usage: float) -> Tuple[int, int]:
    """根据内存使用情况动态调整并发和批次大小"""
    if memory_usage > 90:
        return 5, 10
    elif memory_usage > 85:
        return 10, 20
    else:
        return MAX_CONCURRENT_REQUESTS, BATCH_SIZE

def get_processed_ids() -> Set[Tuple[str, int]]:
    """获取已处理新闻的ID集合"""
    processed = set()
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith('.json'):
            try:
                parts = f.split('_')
                if len(parts) >= 2:
                    company_code = parts[0]
                    news_id = int(parts[1].split('.')[0])
                    processed.add((company_code, news_id))
            except Exception as e:
                logging.warning(f"解析已处理文件出错: {f}, 错误: {str(e)}")
    return processed

def load_news_data_generator(processed_ids: Set[Tuple[str, int]]):
    """生成器方式流式加载数据，跳过已处理记录"""
    for filename in os.listdir(NEWS_DATA_DIR):
        if not filename.endswith('.xlsx'):
            continue

        company_code = filename.split('.')[0]
        filepath = os.path.join(NEWS_DATA_DIR, filename)

        try:
            # 使用低内存模式读取Excel
            with pd.ExcelFile(filepath, engine='openpyxl') as excel:
                df = pd.read_excel(
                    excel,
                    usecols=['日期', '标题', '来源']
                )

            # 日期过滤
            df['日期'] = pd.to_datetime(df['日期'])
            mask = (df['日期'] >= START_DATE) & (df['日期'] <= END_DATE)
            df = df.loc[mask]

            # 逐行生成数据
            for idx, row in df.iterrows():
                news_id = idx + 1
                if (company_code, news_id) not in processed_ids:
                    yield {
                        '日期': row['日期'],
                        '标题': row['标题'],
                        '来源': row['来源'],
                        'company_code': company_code,
                        'id': news_id
                    }

            # 及时释放内存
            del df
            gc.collect()

        except Exception as e:
            logging.error(f"加载文件出错: {filename}, 错误: {str(e)}")

async def analyze_news(news_item: dict, semaphore: asyncio.Semaphore) -> Tuple[dict, dict]:
    """新闻情感分析，带重试机制"""
    async with semaphore:
        prompt = f"""请对以下新闻进行金融情感分析：
标题: {news_item['标题']}
来源: {news_item['来源']}

返回JSON格式：{{
  "sentiment": -10到10的整数,
  "impact": "高/中/低"
}}"""

        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = await aclient.chat.completions.create(
                    model="deepseek-chat",
                    messages=[
                        {"role": "system", "content": "你只需返回情感分数和影响等级"},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.1,
                    max_tokens=20,
                    response_format={"type": "json_object"},
                    timeout=REQUEST_TIMEOUT
                )
                result = json.loads(response.choices[0].message.content)
                return news_item, {
                    "sentiment": int(result.get("sentiment", 0)),
                    "impact": result.get("impact", "中")
                }
            except Exception as e:
                if attempt == max_retries - 1:
                    logging.warning(f"分析失败: {news_item['company_code']}-{news_item['id']}, 错误: {str(e)}")
                    return news_item, {"sentiment": 0, "impact": "中"}
                await asyncio.sleep(2 * (attempt + 1))

def save_results(news_item: dict, analysis: dict) -> None:
    """保存结果到JSON文件"""
    filename = f"{news_item['company_code']}_{news_item['id']}.json"
    filepath = os.path.join(OUTPUT_DIR, filename)

    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump({
                "company_code": news_item["company_code"],
                "date": str(news_item["日期"].date()),
                "title": news_item["标题"],
                "source": news_item["来源"],
                "sentiment": analysis["sentiment"],
                "impact": analysis["impact"]
            }, f, ensure_ascii=False, indent=2)
    except Exception as e:
        logging.error(f"保存结果失败: {filename}, 错误: {str(e)}")

async def process_news_batch(batch: list, pbar: tqdm) -> int:
    """处理新闻批次，返回成功处理数量"""
    memory_usage = get_memory_usage()
    current_concurrency, _ = adjust_parameters(memory_usage)
    semaphore = asyncio.Semaphore(current_concurrency)

    tasks = [analyze_news(news, semaphore) for news in batch]
    processed_count = 0

    for future in asyncio.as_completed(tasks):
        try:
            news_item, analysis = await future
            save_results(news_item, analysis)
            processed_count += 1
            pbar.update(1)

            # 及时释放内存
            del news_item, analysis
            if get_memory_usage() > 85:
                gc.collect()

        except Exception as e:
            logging.error(f"处理批次出错: {str(e)}")

    return processed_count

async def main():
    ensure_output_dir()
    logging.info("程序启动")

    # 获取已处理记录
    processed_ids = get_processed_ids()
    initial_count = len(processed_ids)
    logging.info(f"发现已处理记录: {initial_count}条")

    # 统计剩余记录
    remaining_count = sum(1 for _ in load_news_data_generator(processed_ids))
    if remaining_count == 0:
        logging.info("没有需要处理的新数据")
        print("所有数据已处理完成！")
        return

    print(f"断点续传: 已处理 {initial_count}条 | 待处理 {remaining_count}条")
    logging.info(f"待处理数据量: {remaining_count}条")

    start_time = time.time()
    total_processed = initial_count

    # 初始化进度条
    with tqdm(total=remaining_count + initial_count, initial=initial_count,
              desc="总体进度", position=0) as pbar:

        # 批次处理
        batch_size = BATCH_SIZE
        current_batch = []
        news_generator = load_news_data_generator(processed_ids)

        for news_item in news_generator:
            current_batch.append(news_item)

            # 当批次达到预定大小或内存紧张时处理
            if len(current_batch) >= batch_size or get_memory_usage() > 88:
                processed = await process_news_batch(current_batch, pbar)
                total_processed += processed

                # 动态调整
                _, batch_size = adjust_parameters(get_memory_usage())
                current_batch = []

                # 内存紧张时暂停
                if get_memory_usage() > 90:
                    pause_time = 10
                    logging.warning(f"内存使用高({get_memory_usage()}%), 暂停{pause_time}秒")
                    await asyncio.sleep(pause_time)

        # 处理最后一批
        if current_batch:
            processed = await process_news_batch(current_batch, pbar)
            total_processed += processed

    # 统计最终结果
    duration = time.time() - start_time
    logging.info(f"处理完成! 总处理量: {total_processed}条, 耗时: {duration:.2f}秒")
    print(f"\n处理完成! 总处理量: {total_processed}条")
    print(f"平均速度: {total_processed/duration:.2f}条/秒")
    print(f"结果目录: {os.path.abspath(OUTPUT_DIR)}")

if __name__ == "__main__":
    try:
        # 设置内存限制(Unix系统)
        try:
            import resource
            soft, hard = resource.getrlimit(resource.RLIMIT_AS)
            resource.setrlimit(resource.RLIMIT_AS, (hard, hard))
        except:
            pass

        asyncio.run(main())
    except KeyboardInterrupt:
        processed = len(get_processed_ids())
        print(f"\n用户中断! 已处理: {processed}条")
        logging.warning(f"用户中断! 已处理: {processed}条")
    except Exception as e:
        processed = len(get_processed_ids())
        print(f"\n程序出错! 已处理: {processed}条, 错误: {str(e)}")
        logging.error(f"程序出错! 已处理: {processed}条, 错误: {str(e)}")
    finally:
        # 最终内存状态记录
        mem = psutil.virtual_memory()
        logging.info(
            f"内存使用: 已用{mem.used/1024/1024:.2f}MB, "
            f"可用{mem.available/1024/1024:.2f}MB, "
            f"使用率{mem.percent}%"
        )

所有数据已处理完成！


In [2]:
print(len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json')]))

50957


In [15]:
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime

def load_and_prepare_data():
    # 读取交易日数据（处理单行逗号分隔的情况）
    with open('trade_days.csv', 'r') as f:
        trade_days = f.read().strip().split(',')

    trade_days = [datetime.strptime(day.strip(), '%Y-%m-%d').date() for day in trade_days]
    trade_days.sort()  # 确保日期是升序排列

    # 读取股票列表
    stocks = pd.read_csv('stock_list.csv', header=None)
    stock_codes = stocks[0].tolist()  # 股票代码列表
    stock_names = stocks[1].tolist()  # 股票名称列表
    stock_count = len(stock_codes)

    # 创建股票名称到索引的映射
    name_to_index = {name: idx for idx, name in enumerate(stock_names)}
    code_to_index = {code: idx for idx, code in enumerate(stock_codes)}

    return trade_days, stock_codes, stock_names, name_to_index, code_to_index

def find_trade_day_index(news_date, trade_days):
    """改进的交易日匹配逻辑"""
    if not trade_days:
        return None

    # 处理早于第一个交易日的情况
    if news_date <= trade_days[0]:
        return 0

    # 处理晚于最后一个交易日的情况
    if news_date > trade_days[-1]:
        return len(trade_days) - 1

    # 二分查找提高效率
    left, right = 0, len(trade_days) - 1
    while left <= right:
        mid = (left + right) // 2
        if mid > 0 and trade_days[mid-1] < news_date <= trade_days[mid]:
            return mid
        elif news_date > trade_days[mid]:
            left = mid + 1
        else:
            right = mid - 1

    return None

def process_json_files(json_dir, trade_days, name_to_index, code_to_index, stock_count):
    # 初始化三维张量 (时间, 股票, 特征)
    tensor = np.zeros((len(trade_days), stock_count, 3), dtype=np.float32)

    # 用于调试的统计信息
    date_stats = {
        'min_date': None,
        'max_date': None,
        'matched_counts': np.zeros(len(trade_days), dtype=int),
        'unmatched': 0
    }

    for filename in os.listdir(json_dir):
        if not filename.endswith('.json'):
            continue

        try:
            with open(os.path.join(json_dir, filename), 'r', encoding='utf-8') as f:
                data = json.load(f)

                # 获取股票索引
                company_identifier = data['company_code']
                stock_idx = name_to_index.get(company_identifier)

                # 如果名称不匹配，尝试用代码匹配
                if stock_idx is None:
                    code = company_identifier.replace('.HK', '') + '.HK'
                    stock_idx = code_to_index.get(code)
                    if stock_idx is None:
                        continue

                # 解析日期
                news_date = datetime.strptime(data['date'], '%Y-%m-%d %H:%M:%S').date()

                # 更新日期统计
                if date_stats['min_date'] is None or news_date < date_stats['min_date']:
                    date_stats['min_date'] = news_date
                if date_stats['max_date'] is None or news_date > date_stats['max_date']:
                    date_stats['max_date'] = news_date

                # 找到对应的交易日区间
                trade_day_idx = find_trade_day_index(news_date, trade_days)

                if trade_day_idx is None:
                    date_stats['unmatched'] += 1
                    continue

                date_stats['matched_counts'][trade_day_idx] += 1

                # 更新张量数据
                sentiment = data['sentiment']
                if sentiment > 0:  # 利好新闻
                    tensor[trade_day_idx, stock_idx, 0] += 1
                elif sentiment < 0:  # 利空新闻
                    tensor[trade_day_idx, stock_idx, 1] += 1

                # 累加综合得分
                tensor[trade_day_idx, stock_idx, 2] += sentiment

        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            continue

    return tensor, date_stats

def main():
    # 1. 加载基础数据
    trade_days, stock_codes, stock_names, name_to_index, code_to_index = load_and_prepare_data()

    # 2. 处理JSON文件
    json_dir = 'analysis_results'
    tensor, stats = process_json_files(json_dir, trade_days, name_to_index, code_to_index, len(stock_codes))

    # 3. 输出统计信息和结果
    print("\n===== 处理统计信息 =====")
    print(f"交易日总数: {len(trade_days)}")
    print(f"股票总数: {len(stock_codes)}")
    print(f"JSON日期范围: {stats['min_date']} 到 {stats['max_date']}")
    print(f"匹配到的时间点分布:")
    for idx, count in enumerate(stats['matched_counts']):
        if count > 0:
            print(f"  {trade_days[idx]}: {count}条新闻")
    print(f"未匹配的新闻数量: {stats['unmatched']}")

    print("\n===== 结果张量信息 =====")
    print(f"张量形状: {tensor.shape} (时间点×股票×特征)")

    # 4. 保存结果
    np.save('news_tensor.npy', tensor)
    print("\n处理完成，已保存为 news_tensor.npy")

if __name__ == "__main__":
    main()


===== 处理统计信息 =====
交易日总数: 735
股票总数: 83
JSON日期范围: 2024-01-01 到 2024-03-01
匹配到的时间点分布:
  2024-01-02: 1704条新闻
  2024-01-03: 1146条新闻
  2024-01-04: 1031条新闻
  2024-01-05: 1073条新闻
  2024-01-08: 1399条新闻
  2024-01-09: 1159条新闻
  2024-01-10: 1047条新闻
  2024-01-11: 1139条新闻
  2024-01-12: 1126条新闻
  2024-01-15: 1574条新闻
  2024-01-16: 1124条新闻
  2024-01-17: 1043条新闻
  2024-01-18: 1065条新闻
  2024-01-19: 1099条新闻
  2024-01-22: 1724条新闻
  2024-01-23: 1045条新闻
  2024-01-24: 1088条新闻
  2024-01-25: 1139条新闻
  2024-01-26: 1155条新闻
  2024-01-29: 1534条新闻
  2024-01-30: 1038条新闻
  2024-01-31: 1018条新闻
  2024-02-01: 1149条新闻
  2024-02-02: 1186条新闻
  2024-02-05: 1839条新闻
  2024-02-06: 1044条新闻
  2024-02-07: 1003条新闻
  2024-02-08: 833条新闻
  2024-02-09: 476条新闻
  2024-02-14: 976条新闻
  2024-02-15: 372条新闻
  2024-02-16: 335条新闻
  2024-02-19: 1619条新闻
  2024-02-20: 1129条新闻
  2024-02-21: 1281条新闻
  2024-02-22: 1050条新闻
  2024-02-23: 1123条新闻
  2024-02-26: 1710条新闻
  2024-02-27: 1322条新闻
  2024-02-28: 1366条新闻
  2024-02-29: 1569条新闻
  2024-03-01: 1532

In [16]:
import numpy as np

# 加载npy文件
tensor = np.load("news_tensor.npy")

# 查看基本属性
print("张量形状:", tensor.shape)  # 例如 (60, 83, 3) 表示60个交易日、83只股票、3个特征
print("数据类型:", tensor.dtype)

# 查看第一天的数据（示例）
print("\n第一天所有股票的数据:")
print(tensor[0])  # 形状 (83, 3)

print("\n第二天所有股票的数据:")
print(tensor[1])  # 形状 (83, 3)

print("\n第三天所有股票的数据:")
print(tensor[2])  # 形状 (83, 3)

# 查看某只股票在所有交易日的数据（例如第0号股票）
print("\n某只股票在所有交易日的数据:")
print(tensor[:, 0, :])  # 形状 (60, 3)

# 查看具体值（例如第10天、第5只股票）
print("\n具体某个位置的值:")
print(tensor[10, 5])  # 输出 [利好数, 利空数, 综合得分]

张量形状: (735, 83, 3)
数据类型: float32

第一天所有股票的数据:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0